In [0]:
CREATE OR REPLACE PROCEDURE dbw_healthcare_de_dev.etl.usp_fail_file
(
    IN p_file_id STRING,
    IN p_adf_run_id STRING,
    IN p_failed_stage STRING,
    IN p_error_message STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    UPDATE dbw_healthcare_de_dev.etl.file_control
    SET
        pipeline_status = 'FAILED',
        retry_count = retry_count + 1,
        error_message = p_error_message,
        last_updated_ts = CURRENT_TIMESTAMP(),
        processing_end_ts = CURRENT_TIMESTAMP()

    WHERE file_id = p_file_id
      AND adf_run_id = p_adf_run_id
      AND processing_stage = UPPER(p_failed_stage)
      AND pipeline_status = 'IN_PROGRESS'
      AND UPPER(p_failed_stage) IN ('BRONZE', 'SILVER', 'GOLD');

END;